In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import glob
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# Define transformations for training and testing datasets
transform_train = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

transform_test = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])


class PotatoDataset(Dataset):
    def __init__(self, root_dir, transform=None, split="train"):
        self.root_dir = root_dir
        self.transform = transform
        self.split = split

        self.class_to_idx = {}
        self.idx_to_class = {}

        self.image_paths = []
        self.labels = []

        split_dir = os.path.join(root_dir, split)


        class_names = ['Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy']


        for i, class_name in enumerate(class_names):
            self.class_to_idx[class_name] = i
            self.idx_to_class[i] = class_name



        # Iterate through each class folder to find img
        for class_folder_name in class_names:
            label = self.class_to_idx[class_folder_name]
            class_path = os.path.join(split_dir, class_folder_name)
            class_images = glob.glob(os.path.join(class_path, "*.jpg"))
            class_images.extend(glob.glob(os.path.join(class_path, "*.JPG")))

            self.image_paths.extend(class_images)
            self.labels.extend([label] * len(class_images))


    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label = self.labels[idx]

        image = Image.open(image_path).convert("RGB") # Ensure image is RGB

        if self.transform:
            image = self.transform(image)

        return image, label

dataset_root = os.path.join(path, "PlantVillage")


train_dataset = PotatoDataset(root_dir=dataset_root, transform=transform_train, split="train")
test_dataset = PotatoDataset(root_dir=dataset_root, transform=transform_test, split="test") # Assuming "test" split exists


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0) # No shuffle for test loader

print(f"Total training images: {len(train_dataset)}")
print(f"Total testing images: {len(test_dataset)}")
print(f"Number of training batches: {len(train_loader)}")
print(f"Number of testing batches: {len(test_loader)}")


# Display some  images
fig, axes = plt.subplots(1, 5, figsize=(15, 3))

for i in range(5):
    img, label = train_dataset[i]  # Load image & label



    img = img.numpy().transpose(1, 2, 0)# (C, H, W) -> (H, W, C)


    axes[i].imshow(img)
    axes[i].set_title(train_dataset.idx_to_class[label])
    axes[i].axis('off')

plt.suptitle('Sample Training Images', fontsize=16)
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNNModel(nn.Module):
    def __init__(self, num_classes=3):
        super(CNNModel, self).__init__()
        # Input: 3x32x32
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)


        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.conv4 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)

        self.conv5 = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(512)

        self.fc1 = nn.Linear(512 * 1 * 1, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        # Conv Layer 1
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.max_pool2d(x, kernel_size=2, stride=2)

        # Conv Layer
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.max_pool2d(x, kernel_size=2, stride=2)

        # Conv Layer
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.max_pool2d(x, kernel_size=2, stride=2)

        # Conv Layer
        x = F.relu(self.bn4(self.conv4(x)))
        x = F.max_pool2d(x, kernel_size=2, stride=2)

        # Conv Layer
        x = F.relu(self.bn5(self.conv5(x)))
        x = F.max_pool2d(x, kernel_size=2, stride=2)

        # Flatten the feature maps
        x = x.view(x.size(0), -1)

        # Fully connected layers
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x



In [ ]:
import torch.optim as optim
from tqdm.notebook import tqdm

#  training loop function
def train(model, device, train_loader, optimizer, criterion, epoch):
    model.train() # train
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0


    for batch_idx, (data, target) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch} Training")):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Calculate accuracy
        _, predicted = torch.max(output.data, 1)
        total_samples += target.size(0)
        correct_predictions += (predicted == target).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = correct_predictions / total_samples

    print(f'\nTrain Epoch: {epoch} | Average Loss: {epoch_loss:.4f} | Accuracy: {epoch_accuracy:.4f}')
    return epoch_loss, epoch_accuracy

# Define the valida loop
def validate(model, device, test_loader, criterion):
    model.eval() # Set the model to eval
    test_loss = 0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item()

            # Calculate accuracy
            _, predicted = torch.max(output.data, 1)
            total_samples += target.size(0)
            correct_predictions += (predicted == target).sum().item()

    test_loss /= len(test_loader)
    accuracy = correct_predictions / total_samples

    print(f'Validation set: Average loss: {test_loss:.4f}, Accuracy: {accuracy:.4f}')
    return test_loss, accuracy



In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Set up device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


num_classes = len(train_dataset.class_to_idx)
model = CNNModel(num_classes=num_classes).to(device)
print(f"Model architecture:\n{model}")

# loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training parameters
num_epochs = 10

# Lists to store metrics for plotting
train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []


for epoch in range(1, num_epochs + 1):
    train_loss, train_accuracy = train(model, device, train_loader, optimizer, criterion, epoch)
    val_loss, val_accuracy = validate(model, device, test_loader, criterion)

    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)



# Plot
epochs_range = range(1, num_epochs + 1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_losses, label='Training Loss')
plt.plot(epochs_range, val_losses, label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, train_accuracies, label='Training Accuracy')
plt.plot(epochs_range, val_accuracies, label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# Write your code here
